# Keras Sequential API の紹介

**学習目標**
  1. Keras Sequential API を使用して DNN モデルを構築する
  1. Keras でモデルをトレーニングする方法を学ぶ
  1. GCP 上で Keras モデルを保存/読み込み、デプロイする方法を学ぶ
  1. Keras モデルをデプロイして予測を行う方法を学ぶ

## はじめに

[Keras Sequential API](https://keras.io/models/sequential/) を使用すると、TensorFlow モデルをレイヤごとに構築できます。これはほとんどの種類の機械学習モデルの構築に役立ちますが、レイヤを共有したり、レイヤを再利用したり、複数の入力や出力を持つモデルを作成することはできません。

このラボでは、Keras Sequential API を使用してシンプルなディープ ニューラル ネットワーク モデルを構築する方法を説明します。モデルをトレーニングした後、Agent Platform を使用してデプロイし、オンライン予測のためにモデルを呼び出す方法を確認します。

まず、このラボに必要なライブラリをインポートします。

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"


import datetime
import shutil

import keras
import numpy as np
import pandas as pd
import tensorflow as tf  # tf.data ローダーのみに使用
from google.cloud import aiplatform
from keras import Input
from keras.callbacks import TensorBoard
from keras.layers import Dense
from keras.metrics import RootMeanSquaredError
from keras.models import Sequential
from matplotlib import pyplot as plt

print(keras.__version__)
%matplotlib inline

## 生データの読み込み

この一連の最初のノートブックで作成した CSV ファイルを使用して、taxifare データセットを使用します。これらのファイルは `../data` に保存されています。

In [ ]:
!ls -l ../data/*.csv

In [ ]:
!head ../data/taxi*.csv

## tf.data を使用して CSV ファイルを読み込む

上記の CSV ファイルからデータを読み込むためのこれらの関数は、[前のノートブック](./2a_dataset_api.ipynb) で作成しました。

In [ ]:
def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(ds[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    features = tf.strings.to_number(ds[2:6])  # 一部の特徴量のみを使用
    return features, label


def create_dataset(pattern, batch_size, mode="eval"):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(parse_csv).repeat()
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

## シンプルな Keras DNN モデルの構築

次に、DNN モデルを作成します。Sequential モデルはレイヤの線形スタックであり、Sequential API を使用してモデルを構築する際は、モデルの各レイヤを順番に構成します。すべてのレイヤが追加されたら、モデルをコンパイルします。

**演習:** Keras の Sequential API を使用してディープ ニューラル ネットワークを作成します。下のセルで、`tf.keras.layers` ライブラリを使用してディープ ニューラル ネットワークのすべてのレイヤを作成してください。

In [ ]:
# Sequential API を使用して Keras DNN モデルを構築
model =  # TODO: ここにコードを記述してください

次に、トレーニングに向けてモデルを準備するために、学習プロセスを構成する必要があります。これは compile メソッドを使用して行います。compile メソッドは 3 つの引数を取ります。

* オプティマイザ（Optimizer）: 既存のオプティマイザの文字列識別子（`rmsprop` や `adagrad` など）、または [Optimizer クラス](https://keras.io/api/optimizers/) のインスタンス。
* 損失関数（Loss function）: モデルが最小化しようとする目標。既存の損失関数の文字列識別子（[Losses クラス](https://keras.io/api/losses/) の `categorical_crossentropy` や `mse` など）、またはカスタムの目的関数。
* 評価指標（Metrics）のリスト: あらゆる機械学習の問題において、モデルを評価するための指標セットが必要になります。評価指標は、既存の指標の文字列識別子またはカスタムの評価指標関数です。

**演習:** 上で作成したモデルをコンパイルします。損失関数として `mse` を、評価指標として `rmse` を使用してください。

In [ ]:
# Keras モデルをコンパイル
# TODO: ここにコードを記述してください

## モデルのトレーニング

モデルをトレーニングするために、Keras は次の 2 つの関数を提供しています。
 1. `.fit()`: 固定のエポック数（データセットに対するイテレーション）でモデルをトレーニングします。
 2. `.train_on_batch()`: データの単一バッチに対して単一の勾配更新を実行します。
 
`.fit()` 関数は、NumPy 配列、テンソルのリスト、`tf.data` データセット、Python ジェネレータなど、さまざまな形式のデータで動作します。`.train_on_batch()` メソッドは、トレーニングをよりきめ細かく制御するためのもので、データの単一バッチのみを受け入れます。

上記の `create_dataset` 関数はトレーニング サンプルのバッチを生成するため、`.fit` を使用できます。

まず、トレーニング ジョブのパラメータをいくつか設定し、トレーニング データと検証データのデータセットを作成します。

In [ ]:
BATCH_SIZE = 64
NUM_TRAIN_EXAMPLES = 10000 * 10  # トレーニング データセットはリピート（ラップアラウンド）します
NUM_EVALS = 10  # 評価を行う回数
NUM_EVAL_EXAMPLES = 1000  # 妥当なサンプルを得るのに十分な数

trainds = create_dataset(
    pattern="../data/taxi-train.csv", batch_size=BATCH_SIZE, mode="train"
)

evalds = create_dataset(
    pattern="../data/taxi-valid.csv", batch_size=BATCH_SIZE, mode="eval"
).take(NUM_EVAL_EXAMPLES // BATCH_SIZE)

print(f"Training dataset object: {trainds}")
print(f"Evaluation dataset object: {evalds}")

Keras の `fit` メソッドを呼び出す際に設定できる引数はさまざまです（[ドキュメント](https://keras.io/api/models/model_training_apis/#fit-method) を参照）。主なパラメータは次のとおりです。
- `x`: 入力データ。今回のケースでは、(features, labels) のバッチを生成する `tf.data.Dataset` オブジェクトである `trainds` です。Keras 3 は、TensorFlow 以外のバックエンドでも `tf.data.Dataset` 入力を処理できます。その他の入力タイプについては [ドキュメント](https://keras.io/api/models/model_training_apis/#fit-method) を参照してください。
- `epochs`: トレーニング データセット全体を反復処理する回数。ここでは 10 エポック分トレーニングします。
- `validation_data`: 各エポックの終わりに損失およびモデルの評価指標を評価するためのデータ。今回のケースでは、もう 1 つの `tf.data.Dataset` オブジェクトである `evalds` です。
- `callbacks`: トレーニング中に適用する [コールバック](https://keras.io/api/callbacks/) のリスト。ここでは TensorBoard で可視化するためのログを保存するために `TensorBoard` コールバックを使用します。

**演習:** 下のセルでモデルをトレーニングします。`.fit()` を使用してモデルをトレーニングし、モデルのトレーニング出力を `history` という名前の変数に保存してください。

In [ ]:
%%time
steps_per_epoch = NUM_TRAIN_EXAMPLES // (BATCH_SIZE * NUM_EVALS)

LOGDIR = "./taxi_trained"
history =  # TODO: ここにコードを記述してください

### 高レベルのモデル評価

モデルにデータを通したら、モデルに対して `.summary()` を呼び出して、ネットワークの概要を取得できます。また、上記で計算した評価指標のトレーニング曲線と検証曲線をプロットすることもできます。

In [ ]:
model.summary()

`.fit` を実行すると、トレーニング中に記録されたすべてのイベントを収集した History オブジェクトが返されます。TensorBoard と同様に、History オブジェクトのこれらの要素にアクセスすることで、モデルの損失と RMSE のトレーニング曲線および検証曲線をプロットできます。

まだ特徴量エンジニアリングを一切行っていないため、パフォーマンスはあまり良くないかもしれません。これは次のノートブックで対処します。

In [ ]:
RMSE_COLS = ["root_mean_squared_error", "val_root_mean_squared_error"]

pd.DataFrame(history.history)[RMSE_COLS].plot()

In [ ]:
LOSS_COLS = ["loss", "val_loss"]

pd.DataFrame(history.history)[LOSS_COLS].plot()

# モデルによる予測の実行

トレーニング済みモデルで予測を行うには、`predict` メソッドを呼び出します（[ドキュメント](https://keras.io/api/models/model_training_apis/#predict-method) を参照）。

下のコードセルに示すように、モデル オブジェクトを直接呼び出す場合は、入力特徴量を含む NumPy 配列を渡します。イテレータが渡された場合は `steps` パラメータを使用できますが、直接 NumPy 配列を入力する場合、少数の予測では通常必要ありません。

注: 後でこのモデルを Agent Platform エンドポイントにデプロイする場合、エンドポイントへの予測リクエストの入力形式は通常 JSON 辞書になり、ここでの直接の `model.predict()` 呼び出しとは異なる場合があります。

In [ ]:
model.predict([[-73.982683, 40.742104, -73.983766, 40.755174]])

# モデルのエクスポートとデプロイ

他の人がトレーニング済みモデルを使用できるようにするため（特に本番環境で）、モデルを保存（または「エクスポート」）する必要があります。Keras 3 では、アーキテクチャ、重み、オプティマイザの状態を 1 つのファイルに保存する `model.save('my_model.keras')` という新しい統合されたモデル保存方法が提供されています（[Keras の保存とシリアル化のガイド](https://keras.io/guides/serialization_and_saving/) を参照）。

ただし、TensorFlow Serving コンテナを使用して Agent Platform などの特定のプラットフォームにデプロイする場合は、TensorFlow SavedModel 形式が推奨されることがよくあります。Keras 3 で TensorFlow バックエンドを使用している場合は、次のセルに示すように `model.export(filepath)` を使用してこの形式にエクスポートできます。別のバックエンド（JAX または PyTorch）を使用している場合でも、Keras モデルをこの形式でシリアル化できますが、デプロイ構成やインフラストラクチャが若干異なる場合があります。

SavedModel 形式にエクスポートした後、`saved_model_cli` ツールを使用してそのシグネチャを検査できます。これにより、サービングに必要な入力および出力のテンソル名と形状を理解するのに役立ちます。

In [ ]:
OUTPUT_DIR = "./export"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(OUTPUT_DIR, TIMESTAMP)

SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")
KERAS_PATH = os.path.join(EXPORT_PATH, "model.keras")

model.export(SAVEDMODEL_PATH)
model.save(KERAS_PATH)

モデルを Keras モデル オブジェクトとして読み込みたい場合は、`keras.models.load_model` と `.keras` シリアル化形式を使用します。

In [ ]:
loaded_model = keras.models.load_model(KERAS_PATH)
loaded_model.summary()

`saved_model_cli` を使用して、保存した SavedModel を検査してみましょう。

In [ ]:
!saved_model_cli show \
    --tag_set serve \
    --signature_def serving_default \
    --dir # TODO: ここにコードを記述してください

!find {EXPORT_PATH}
os.environ['EXPORT_PATH'] = EXPORT_PATH

## Agent Platform へのモデルのデプロイ

最後に、トレーニングしたモデルを Google Cloud の Agent Platform にデプロイし、オンライン予測を行う方法を確認します。

まず、サービング コンテナ イメージとともにモデルをアップロードする必要があります。Agent Platform は、[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers) に記載されているサービング用の構築済み Docker イメージをホストしています。

**重要な注意点:** Agent Platform にデプロイするための以下の手順は、TensorFlow バックエンドを前提としています。

In [ ]:
PROJECT = !gcloud config get-value project 2>/dev/null
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"
MODEL_DISPLAYNAME = f"taxifare-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

# from https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
)
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
# GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gcloud storage ls
fi

In [ ]:
!gcloud storage cp -R $SAVEDMODEL_PATH gs://$BUCKET/$MODEL_DISPLAYNAME

## モデルのアップロード

`aiplatform.Model.upload` メソッドはモデルを Agent Platform Model Registry に登録し、検出可能でデプロイ可能なモデル リソースを作成します。

**演習:** 下のセルのコードを完成させて、`Model.upload` メソッドを使用してトレーニングしたモデルを Agent Platform にアップロードしてデプロイします。[ドキュメント](https://googleapis.dev/python/aiplatform/latest/aiplatform.html#google.cloud.aiplatform.Model) を参照してください。

In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAYNAME,
    artifact_uri=  # TODO: ここにコードを記述してください
    serving_container_image_uri=  # TODO: ここにコードを記述してください
)

## エンドポイントの作成とデプロイ

アップロード後、`uploaded_model.deploy` はエンドポイントを作成し、そこにモデルをデプロイします。エンドポイントは、予測を配信するための専用の Agent Platform リソースです。パラメータには、machine_type（マシン リソース）や accelerator_type/accelerator_count（GPU アクセラレーション用）が含まれます。

**デプロイには約 10 分かかります。**

In [ ]:
MACHINE_TYPE = "n1-standard-2"

endpoint = uploaded_model.deploy(
    machine_type=MACHINE_TYPE,
    accelerator_type=None,
    accelerator_count=None,
)

デプロイされると、モデルはオンライン予測リクエストにアクセスできるようになります。

この JSON オブジェクト内のキーは、モデルのサービング シグネチャ内の入力テンソル名と一致している必要があります。
たとえば、モデルの入力レイヤの名前が `'input_layer'` である場合、JSON 入力は `{"input_layer": [...]}` のようになり、各データ ポイントの形状は (4,) である必要があります。これにより、予測サービスが入力データを想定されるモデル入力に正しくマッピングできるようになります。

In [ ]:
instances = [
    {"input_layer": [-73.982683, 40.742104, -73.983766, 40.755174]},
    {"input_layer": [-73.976738, 40.751321, -73.986671, 40.748831]},
]

**演習:** 下のセルのコードを完成させて、上で `instances` 変数に作成したサンプルに対して、デプロイされたモデルで予測を呼び出します。

In [ ]:
endpoint.predict(
    # TODO: ここにコードを記述してください
)

# クリーンアップ

モデルをオンライン予測用のエンドポイントにデプロイする場合、最小の `min-replica-count` は 1 であり、ノード時間ごとに課金されます。そのため、不要な課金を減らすためにエンドポイントを削除しましょう。エンドポイントを削除する前に、まず接続されているすべてのモデルのデプロイを解除（undeploy）します...

In [ ]:
endpoint.undeploy_all()

...その後、エンドポイントを削除します。

In [ ]:
endpoint.delete()

Copyright 2021 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.